In [1]:
import pandas as pd

In [6]:
years = [2019, 2020, 2021]
sheets = ["Table 218", "Table 219", "Table 220"]
dfs = []
for index, sheet in enumerate(sheets):
    dfs.append(pd.read_excel("../AGRICULTURE/anambra_2022_year_book.xlsx", sheet_name=sheets[index]))

lga_df = pd.read_csv("../Geography/Anambra_LGAs.csv")
lga_df.head()

,lga_id,lga_name,zone,AREA_SQ_KM,HIGHLANDS_percent,PLAIN_LANDS_percent
0,1,ONITSHA SOUTH,ONITSHA,196,20,48
1,2,ANAMBRA WEST,OTUOCHA,613,15,21
2,3,IDEMILI SOUTH,OGIDI,145,20,25
3,4,NNEWI SOUTH,NNEWI,176,34,38
4,5,OYI,OGIDI,331,10,40


In [7]:
for index, df in enumerate(dfs):
    dfs[index] = dfs[index].fillna(0)
    dfs[index].replace(['NA', 'NIL'], 0, inplace=True)

dfs[1].head()

,LGA,WATER PROJECT UNDERTAKEN,NUMBER OF TOWNS COVERED,TOTAL VOLUME OF WATER RELEASED DAILY (MILLION/LITRE)
0,AGUATA,Obizi Regional Water Supply Scheme,"Uga, Ekwulobia, Ezinifite, Isuofia, Igboukwu","400,000 litres/day"
1,ANAMBRA EAST,Otuocha Regional Water Scheme,"Aguleri, Umueri and Umuoba -Anam","400,000 litres/day"
2,ANAMBRA WEST,0,0,0
3,ANAOCHA,0,0,0
4,AWKA NORTH,0,0,0


In [29]:
columns = ['lga_id', 'project', 'year', 'towns_covered', 'count']
def count_of_towns_covered(towns):
    towns = towns.replace("and", ',')
    town_count = towns.split(',')
    count = 0
    for elem in town_count:
        if(len(elem.strip()) > 1): count+=1
    return count

new_df = pd.DataFrame(columns=columns)
new_df["towns_covered"] = new_df["towns_covered"].astype(str)
for i, df in enumerate(dfs):
    year = years[i]
    for index, row in df.iterrows():
        if(len(lga_df.loc[lga_df['lga_name'] == row["LGA"], "lga_id"].values) == 0):
            print(row["LGA"])

        transformed_data = [{
            'lga_id': lga_df.loc[lga_df['lga_name'] == row["LGA"], "lga_id"].values[0],
            'project': row["WATER PROJECT UNDERTAKEN"] if row["WATER PROJECT UNDERTAKEN"] != 0.0 else "",
            'year': year,
            'towns_covered': row["NUMBER OF TOWNS COVERED"] if row["NUMBER OF TOWNS COVERED"] != 0.0 else "",
            'no_of_towns_covered': count_of_towns_covered(str(row["NUMBER OF TOWNS COVERED"] if row["NUMBER OF TOWNS COVERED"] != 0.0 else "")),
            'count': int(row["TOTAL VOLUME OF WATER RELEASED DAILY (MILLION/LITRE)"].replace("litres/day", "").replace(",", "").strip()) if row["TOTAL VOLUME OF WATER RELEASED DAILY (MILLION/LITRE)"] != 0.0 else "0",
        }]   

        new_df = pd.concat([new_df, pd.DataFrame(transformed_data)], ignore_index=True)


new_df.head(22)

,lga_id,project,year,towns_covered,count,no_of_towns_covered
0,13,,2019,,0,0.0
1,18,,2019,,0,0.0
2,2,,2019,,0,0.0
3,9,,2019,,0,0.0
4,15,,2019,,0,0.0
5,20,,2019,,0,0.0
6,6,,2019,,0,0.0
7,12,,2019,,0,0.0
8,17,,2019,,0,0.0
9,8,,2019,,0,0.0


In [31]:
new_df['count']=new_df['count'].astype(int)
new_df['no_of_towns_covered']=new_df['no_of_towns_covered'].astype(int)

new_df.head()

,lga_id,project,year,towns_covered,count,no_of_towns_covered
0,13,,2019,,0,0
1,18,,2019,,0,0
2,2,,2019,,0,0
3,9,,2019,,0,0
4,15,,2019,,0,0


In [32]:
new_df.to_excel("portable_water_supply_projects.xlsx", index=False)
print("Done!")

Done!


In [33]:
import numpy as np
print(np.sum(new_df["count"]))

2900000
